# Group 2 — Body awareness (postural sway)

Load your **force-plate** data and plot the **sway** — how much your centre of pressure (CoP) wanders. You compare eyes-open vs eyes-closed (and your takes) yourself.

Each take has **two plates** (`_forceplate_1`, `_forceplate_2`) = your two feet.

In [ ]:
!pip install -q pysampled huggingface_hub

### 1 · Get a take

In [ ]:
from huggingface_hub import hf_hub_download
REPO = "praneethnamburimit/immersionlab-pe-mis-groups"
# takes: BodyAwarenesss_Tak5JCH, _Take2, _Take3, _Take4JCH  (each _forceplate_1 and _2)
csv = hf_hub_download(REPO, "group_2/BodyAwarenesss_Take3_forceplate_1.csv", repo_type="dataset")
print(csv)

### 2 · The force-plate loader
This is **`bertec.MotiveLog`** — it isn't on pip, so it's pasted here. Run this cell once.

In [ ]:
import csv as _csv, numpy as np, pandas as pd, pysampled
class MotiveLog:
    """Load a Motive force-plate 'Device export' CSV (from immersionlab.motive_forceplate.Log)."""
    def __init__(self, fname):
        hdr = {}
        with open(fname, newline='') as f:
            for rc, row in enumerate(_csv.reader(f)):
                if not row: continue
                if row[0] == 'MocapFrame': break
                v = row[1] if len(row) > 1 else None
                try: hdr[row[0]] = float(v)
                except (TypeError, ValueError): hdr[row[0]] = v
        self.data = pd.read_csv(fname, skiprows=rc).rename(columns=lambda x: x.strip())
        self.sr = hdr['Mocap Rate'] * hdr.get('Mocap Rate Multiple', 1)
    def __getattr__(self, k):
        if k in ('Fx','Fy','Fz','Mx','My','Mz','Cx','Cy','Cz','Tz'):
            return pysampled.Data(self.data[k].values, sr=self.sr)
        raise AttributeError(k)

### 3 · Load it + grab the centre of pressure

In [ ]:
fp = MotiveLog(csv)
print("sample rate:", fp.sr, "Hz")
cx = np.asarray(fp.Cx()).ravel()   # CoP left-right
cy = np.asarray(fp.Cy()).ravel()   # CoP fore-aft
print("samples:", len(cx))

### 4 · Plot the sway (stabilogram)

In [ ]:
import matplotlib.pyplot as plt
x = cx - np.nanmean(cx); y = cy - np.nanmean(cy)
path = np.nansum(np.hypot(np.diff(x), np.diff(y)))           # total sway path
cov = np.cov(x, y); ev, V = np.linalg.eigh(cov)
area = np.pi*5.991*np.sqrt(max(ev[0]*ev[1], 0))             # 95% confidence ellipse
plt.figure(figsize=(6,6)); plt.plot(x, y, lw=0.5); plt.gca().set_aspect("equal")
th = np.linspace(0, 2*np.pi, 100)
e = V @ (np.sqrt(5.991*ev)[:,None] * np.vstack([np.cos(th), np.sin(th)]))
plt.plot(e[0], e[1], "r--", label="95% ellipse")
plt.title(f"sway path {path:.0f} · ellipse {area:.0f}"); plt.xlabel("CoP X"); plt.ylabel("CoP Y"); plt.legend(); plt.show()

### 5 · Your turn
- Each take / condition is a different file. Load the ones you want (eyes-open vs eyes-closed, …) and compare the **path length** and **ellipse area** — bigger = more sway.
- You have two plates per take (your two feet) — `_forceplate_1` and `_forceplate_2`.
- `fp.Fz()` gives vertical force; `fp.CoP` is the raw pysampled signal if you want to filter it.